# GSEA dot plot — Clinical Outcome (Good vs Poor)

Plots the preranked-GSEA CSVs written by `gsea_clinical_outcome.ipynb` into
`outputs/gsea/clinical_outcome/`.

**One database at a time.** Set `DB_NAME` in the config cell to one of
`hallmark`, `BP`, `CC`, `MF`, `reactome`, `KEGG_medicus`, `oncogenic_sigs`, then
re-run. Nothing loops over databases, and the figure filename carries `DB_NAME`,
so switching databases never overwrites another database's figure.

**Direction convention (important).** Genes were ranked by the Wald `stat` of the
clinical-outcome contrast, so the sign of the NES carries the biology:

| NES | Meaning | Colour |
|---|---|---|
| positive | pathway **Higher in Good** (5Plus) | green `#2c9d2a` |
| negative | pathway **Higher in Poor** (2Minus) | red `#E63946` |

Terms are called significant at `fdr < FDR_CUTOFF` (0.05 — deliberately stricter
than the conventional GSEA 0.25).

**Output:** a single SVG, 600 dpi, editable text, to
`outputs/images/clinical_outcome/gsea_{DB_NAME}_dotplot_clinical_outcome.svg`.

> **Fonts.** Arial is *not* installed on this machine, so `font.family` lists
> `['Arial', 'Liberation Sans']`. Liberation Sans is metrically identical to Arial, so
> matplotlib lays the text out with Arial-correct widths, while the SVG declares the
> font stack `'Arial', 'Liberation Sans'` — Illustrator (or a journal's typesetter)
> renders true Arial where it exists. Setting `'Arial'` alone would silently lay out
> with DejaVu Sans metrics instead.

> This notebook only ever **reads** the GSEA CSVs. It never touches the `.h5ad`.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# necessary packages
import os
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns

## Configuration

`DB_NAME` is the only thing to change when moving to another database.

In [ ]:
# ---- pick ONE database here -------------------------------------------------
DB_NAME = 'hallmark'    # 'hallmark' | 'BP' | 'CC' | 'MF' | 'reactome' | 'KEGG_medicus' | 'oncogenic_sigs'

DB_FILES = {
    'hallmark':       'enriched_hallmark.csv',
    'BP':             'enriched_BP.csv',
    'CC':             'enriched_CC.csv',
    'MF':             'enriched_MF.csv',
    'reactome':       'enriched_reactome.csv',
    'KEGG_medicus':   'enriched_KEGG_medicus.csv',
    'oncogenic_sigs': 'enriched_oncogenic_sigs.csv',
}

GSEA_DIR = f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/gsea/clinical_outcome'
IMG_DIR  = f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome'

CSV_PATH = os.path.join(GSEA_DIR, DB_FILES[DB_NAME])

FDR_CUTOFF = 0.05    # stricter than the usual GSEA 0.25 - deliberate
TOP_N      = 20      # terms per direction, full figure
TRIM_N     = 7       # terms per direction, trimmed figure -> 14 total

# fixed project brand colours for the clinical-outcome contrast
GOOD_COLOR = '#2c9d2a'   # positive NES -> higher in Good (5Plus)
POOR_COLOR = '#E63946'   # negative NES -> higher in Poor (2Minus)
DIR_COLORS = {'Higher in Good': GOOD_COLOR, 'Higher in Poor': POOR_COLOR}

# sequential, perceptually uniform; low fdr (more significant) renders dark
FDR_CMAP = sns.color_palette('mako', as_cmap=True)

# prefixes stripped from term names for display
DB_PREFIXES = ('HALLMARK_', 'GOBP_', 'GOCC_', 'GOMF_', 'REACTOME_', 'KEGG_MEDICUS_')
LABEL_MAXLEN = 55    # truncate very long term labels

os.makedirs(IMG_DIR, exist_ok=True)

# ---- fonts ------------------------------------------------------------------
# Arial is NOT installed here. Liberation Sans is, and is metrically identical to
# Arial (same advance widths). Listing both means:
#   * matplotlib resolves metrics from LiberationSans-Regular.ttf -> Arial-correct layout
#   * svg.fonttype='none' writes the stack "'Arial', 'Liberation Sans'" into the SVG,
#     so Illustrator/journals with Arial render true Arial, others fall back cleanly.
# Setting font.family='Arial' alone would silently lay out with DejaVu Sans (~8% wider).
plt.rcParams['font.family'] = ['Arial', 'Liberation Sans']
plt.rcParams['svg.fonttype'] = 'none'      # keep SVG text editable in Illustrator/Inkscape

# 'Arial' is deliberately first so it lands in the SVG's font-family, but since it is not
# installed here font_manager logs "findfont: Font family 'Arial' not found" for EVERY text
# object (~200 lines/run). The lookup then falls through to Liberation Sans, which is exactly
# what we want, so the message is noise rather than a problem - silence it.
# (This is a logging call, not a warning, so warnings.filterwarnings does not catch it.)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# ---- publication font sizes -------------------------------------------------
plt.rcParams['font.size']            = 14
plt.rcParams['axes.titlesize']       = 16
plt.rcParams['axes.labelsize']       = 15
plt.rcParams['xtick.labelsize']      = 13
plt.rcParams['ytick.labelsize']      = 13
plt.rcParams['legend.fontsize']      = 13
plt.rcParams['legend.title_fontsize'] = 13

print(f'database : {DB_NAME}')
print(f'csv      : {CSV_PATH}')
print(f'figure   : {IMG_DIR}')
print(f'font     : {plt.rcParams["font.family"]}')

## Helpers

In [ ]:
def clean_term(term, maxlen=LABEL_MAXLEN):
    """Strip the MSigDB database prefix, de-underscore and tidy a term for display."""
    label = str(term)
    for prefix in DB_PREFIXES:
        if label.startswith(prefix):
            label = label[len(prefix):]
            break
    label = label.replace('_', ' ').strip().title()
    if maxlen is not None and len(label) > maxlen:
        label = label[:maxlen - 1].rstrip() + '\u2026'
    return label


def save_fig(fig_or_none, name):
    """Save a figure as SVG (600 dpi) into IMG_DIR. Accepts None so a guarded cell can no-op."""
    if fig_or_none is None:
        print(f'[skip] nothing to save for {name}')
        return None
    path = os.path.join(IMG_DIR, name)
    fig_or_none.savefig(path, format='svg', dpi=600, bbox_inches='tight')
    print(f'[saved] {path}')
    return path


def select_top(sig, n_per_direction):
    """Top n terms per direction: fdr ascending, ties broken by |NES| descending.

    The tie-break is not cosmetic. Hundreds of terms share fdr == 0 (reactome: 239
    of them), so ranking on fdr alone would return an arbitrary slice of the tie
    block - "whichever came first in the CSV" rather than the strongest terms.
    Using one ranking for both figures also guarantees the trimmed selection is a
    strict subset of the full one.
    """
    return (sig.assign(abs_nes=sig['nes'].abs())
               .sort_values(['fdr', 'abs_nes'], ascending=[True, False])
               .groupby('direction', group_keys=False)
               .head(n_per_direction)
               .sort_values('nes'))

## Load the selected database

The CSVs are written with a pandas index, hence the `Unnamed: 0` column dropped here.

In [ ]:
gsea = pd.read_csv(CSV_PATH)
gsea = gsea.drop(columns=[col for col in gsea.columns if col.startswith('Unnamed')])

# direction is derived from the sign of the NES - see the header cell
gsea['direction'] = np.where(gsea['nes'] >= 0, 'Higher in Good', 'Higher in Poor')
gsea['label'] = gsea['Term'].apply(clean_term)

sig = gsea[gsea['fdr'] < FDR_CUTOFF].copy()

print(f'{DB_NAME}: {len(gsea)} terms total, {len(sig)} significant at fdr < {FDR_CUTOFF}')
if len(sig):
    print(sig['direction'].value_counts().to_string())
else:
    print('No significant terms - the dot plot below will no-op.')

gsea.head()

## Dot plot

NES on x (so direction is carried by position, not colour alone), terms on y, dot
**size** = `matched_size` (genes matched into the ranking), dot **colour** = FDR on a
perceptually uniform `mako` ramp where *darker = more significant*.

Two figures are written per database:

* **full** — top `TOP_N` (20) per direction, as before;
* **trimmed** — top `TRIM_N` (7) per direction, i.e. **at most 14 terms**, for use as a
  main figure. Only produced when the full figure would exceed 14 terms; otherwise it
  would just duplicate it.

Both use the same `select_top` ranking, so the trimmed panel is always a strict subset
of the full one. Where a direction has fewer than `TRIM_N` significant terms the trimmed
figure simply shows fewer (KEGG_medicus has only 1 significant Good term, so it gets
1 + 7 = 8) rather than padding from the other direction.

In [ ]:
def dot_plot(top, title, fig_name):
    """Draw and save one NES dot plot for an already-selected set of terms."""
    # map matched_size -> marker area
    ms = top['matched_size'].astype(float)
    if ms.max() > ms.min():
        sizes = 50 + (ms - ms.min()) / (ms.max() - ms.min()) * 350
    else:
        sizes = pd.Series(200.0, index=ms.index)

    norm = Normalize(vmin=0, vmax=max(FDR_CUTOFF, float(top['fdr'].max())))

    # geometry grows with the number of terms so the larger type never collides
    fig, ax = plt.subplots(figsize=(11, max(4.0, 0.42 * len(top) + 2.0)))
    ax.scatter(top['nes'], top['label'],
               s=sizes, c=top['fdr'], cmap=FDR_CMAP, norm=norm,
               edgecolor='white', linewidth=0.8, zorder=3)

    ax.axvline(0, color='0.25', linewidth=1, zorder=1)
    ax.grid(axis='y', color='0.9', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlabel('Normalized enrichment score (NES)')
    ax.set_ylabel('')
    ax.set_title(title)

    # colour the y labels by direction so identity is never colour-of-dot alone
    dir_by_label = dict(zip(top['label'], top['direction']))
    for tick in ax.get_yticklabels():
        tick.set_color(DIR_COLORS.get(dir_by_label.get(tick.get_text()), 'black'))

    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=FDR_CMAP), ax=ax, pad=0.02, shrink=0.6)
    cbar.set_label('FDR q-value')
    cbar.ax.invert_yaxis()   # most significant at the top

    # size legend at evenly spaced representative matched_size values
    ticks = np.unique(np.linspace(ms.min(), ms.max(), 3).round().astype(int))
    size_handles = []
    for t in ticks:
        if ms.max() > ms.min():
            s = 50 + (t - ms.min()) / (ms.max() - ms.min()) * 350
        else:
            s = 200.0
        size_handles.append(ax.scatter([], [], s=s, c='0.55', edgecolor='white',
                                       linewidth=0.8, label=str(int(t))))
    leg = ax.legend(handles=size_handles, title='Matched genes', loc='upper left',
                    bbox_to_anchor=(1.16, 1.0), frameon=False, labelspacing=1.4,
                    borderpad=0.9, handletextpad=1.2)
    ax.add_artist(leg)

    dir_handles = [mpatches.Patch(color=DIR_COLORS[d], label=d)
                   for d in ['Higher in Good', 'Higher in Poor'] if (top['direction'] == d).any()]
    ax.legend(handles=dir_handles, loc='lower left', bbox_to_anchor=(1.16, 0.0), frameon=False)

    sns.despine(ax=ax, left=True)
    fig.tight_layout()
    save_fig(fig, fig_name)
    plt.show()


if len(sig) == 0:
    print(f'[{DB_NAME}] no terms with fdr < {FDR_CUTOFF} - nothing to plot.')
    save_fig(None, f'gsea_{DB_NAME}_dotplot_clinical_outcome.svg')
else:
    # ---- full figure ---------------------------------------------------------
    top_full = select_top(sig, TOP_N)
    print(f'[{DB_NAME}] full: {len(top_full)} terms '
          f'({(top_full["direction"] == "Higher in Good").sum()} Good / '
          f'{(top_full["direction"] == "Higher in Poor").sum()} Poor)')
    dot_plot(top_full,
             f'{DB_NAME}: GSEA dot plot (fdr < {FDR_CUTOFF})',
             f'gsea_{DB_NAME}_dotplot_clinical_outcome.svg')

    # ---- trimmed figure ------------------------------------------------------
    # skipped when the full figure is already within the trimmed cap, since the
    # two would then be identical
    if len(top_full) > 2 * TRIM_N:
        top_trim = select_top(sig, TRIM_N)
        print(f'[{DB_NAME}] trimmed: {len(top_trim)} terms '
              f'({(top_trim["direction"] == "Higher in Good").sum()} Good / '
              f'{(top_trim["direction"] == "Higher in Poor").sum()} Poor)')
        dot_plot(top_trim,
                 f'{DB_NAME}: top {TRIM_N} per direction (fdr < {FDR_CUTOFF})',
                 f'gsea_{DB_NAME}_dotplot_trimmed_clinical_outcome.svg')
    else:
        print(f'[{DB_NAME}] full figure has {len(top_full)} terms (<= {2 * TRIM_N}) - '
              f'already trimmed, no separate trimmed figure.')